# Surrogate para política fiscal fixa

Este notebook segue [o plano](fiscal-policy-surrogate-plan.md) para prever **as médias de 12 trimestres** do experimento austríaco na escala `0.001`. A unidade estatística é a **configuração fiscal**; o número de sementes vem dos metadados do experimento. As partições já definidas no dataset são preservadas.

Execute as células de cima para baixo. Defina `EXPERIMENT_NAME` na primeira célula de código para escolher o piloto ou a expansão. Os artefatos são gravados na subpasta `surrogate/` do experimento escolhido.

## 1. Preparar ambiente e localizar o experimento

Instale as dependências de `requirements.txt` no ambiente do kernel. O notebook aceita execução a partir desta pasta ou da raiz do repositório.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import tomllib
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

EXPERIMENT_NAME = "austria-scale-0.001-expanded"  # Use "austria-scale-0.001" para o piloto.
HERE = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "fiscal-policy-surrogate-plan.md").exists()
     and (p / "generate_dataset.jl").exists()),
    None,
)
if HERE is None:
    HERE = next(
        (p / "dba-studies/machine-learning/fiscal-policy-dataset"
         for p in [Path.cwd(), *Path.cwd().parents]
         if (p / "dba-studies/machine-learning/fiscal-policy-dataset"
               / "generate_dataset.jl").exists()),
        None,
    )
if HERE is None:
    raise FileNotFoundError("Abra o notebook nesta pasta ou na raiz do repositório.")
DATA = HERE / "experiments" / EXPERIMENT_NAME
OUTPUT = DATA / "surrogate"
INPUTS = ["tau_INC", "tau_VAT", "theta_UB", "log_m_G"]
METRICS = [
    "real_gdp", "real_household_consumption", "real_capitalformation",
    "unemployment_rate", "gdp_deflator_inflation", "government_revenue",
    "government_debt", "debt_to_gdp", "planned_government_consumption",
    "realized_government_consumption",
]
QUARTERS = list(range(1, 13))
TARGETS = [f"{metric}_q{quarter:02d}" for metric in METRICS for quarter in QUARTERS]
print(DATA)

## 2. Validar os CSVs e montar $X$ e $Y$

Para a configuração $i$, $X_i=(\tau^{INC},\tau^{VAT},\theta^{UB},\log m^G)$ e $Y_i$ concatena 10 indicadores × 12 trimestres. As médias vêm de `summary.csv`; as trajetórias por semente servem para medir o ruído de simulação.

In [ ]:
files = ["configurations.csv", "runs.csv", "observations.csv", "summary.csv", "metadata.toml"]
missing = [name for name in files if not (DATA / name).exists()]
if missing:
    raise FileNotFoundError(f"Arquivos ausentes: {missing}")
with (DATA / "metadata.toml").open("rb") as f:
    experiment_metadata = tomllib.load(f)
N_CONFIGS = int(experiment_metadata["experiment"]["configurations"])
N_SEEDS = int(experiment_metadata["experiment"]["seeds"])
assert experiment_metadata["experiment"]["horizon"] == len(QUARTERS)

configs = pd.read_csv(DATA / "configurations.csv").sort_values("configuration_id").reset_index(drop=True)
runs = pd.read_csv(DATA / "runs.csv")
obs = pd.read_csv(DATA / "observations.csv")
summary = pd.read_csv(DATA / "summary.csv")
ids = configs["configuration_id"].to_numpy()
assert len(ids) == len(np.unique(ids)) == N_CONFIGS
n_holdout = max(1, round(0.1 * N_CONFIGS))
assert configs["split"].value_counts().to_dict() == {"train": N_CONFIGS - 2*n_holdout, "validation": n_holdout, "test": n_holdout}
assert len(runs) == N_CONFIGS*N_SEEDS and runs["status"].eq("success").all()
assert not runs.duplicated(["configuration_id", "seed"]).any()
assert runs.groupby("configuration_id")["seed"].nunique().reindex(ids).eq(N_SEEDS).all()
assert len(summary) == N_CONFIGS*len(QUARTERS) and not summary.duplicated(["configuration_id", "quarter"]).any()
assert summary.groupby("configuration_id")["quarter"].apply(
    lambda q: set(q) == set(QUARTERS)
).reindex(ids).all()
assert summary["successful_seeds"].eq(N_SEEDS).all()
assert len(obs) == N_CONFIGS*N_SEEDS*len(QUARTERS) and not obs.duplicated(["configuration_id", "seed", "quarter"]).any()
assert obs.groupby(["configuration_id", "seed"])["quarter"].apply(
    lambda q: set(q) == set(QUARTERS)
).all()
assert set(runs["configuration_id"]) == set(summary["configuration_id"]) == set(obs["configuration_id"]) == set(ids)
assert runs.groupby("configuration_id")["seed"].apply(lambda x: frozenset(x)).nunique() == 1
assert set(map(tuple, runs[["configuration_id", "seed"]].to_numpy())) == set(
    map(tuple, obs[["configuration_id", "seed"]].drop_duplicates().to_numpy())
)

X = configs[INPUTS].to_numpy(dtype=float)
def matrix_by_quarter(frame, index, suffix=""):
    return np.concatenate([
        frame.pivot(index=index, columns="quarter", values=metric + suffix)
             .reindex(index=pd.Index(ids, name="configuration_id") if index == "configuration_id" else None,
                      columns=QUARTERS)
             .to_numpy(dtype=float)
        for metric in METRICS
    ], axis=1)

Y = matrix_by_quarter(summary, "configuration_id", "_mean")
Y_se = matrix_by_quarter(summary, "configuration_id", "_se")
run_index = runs[["configuration_id", "seed"]].sort_values(
    ["configuration_id", "seed"]
).set_index(["configuration_id", "seed"]).index
Y_runs = np.concatenate([
    obs.pivot(index=["configuration_id", "seed"], columns="quarter", values=metric)
       .reindex(index=run_index, columns=QUARTERS).to_numpy(dtype=float)
    for metric in METRICS
], axis=1)
assert X.shape == (N_CONFIGS, 4) and Y.shape == Y_se.shape == (N_CONFIGS, 120)
assert Y_runs.shape == (N_CONFIGS*N_SEEDS, 120)
assert np.isfinite(X).all() and np.isfinite(Y).all() and np.isfinite(Y_se).all()
assert np.isfinite(Y_runs).all() and (Y_se >= 0).all()
run_means = np.stack([
    Y_runs[run_index.get_level_values("configuration_id") == config_id].mean(axis=0)
    for config_id in ids
])
np.testing.assert_allclose(Y, run_means, rtol=1e-10, atol=1e-10)
split = {name: np.flatnonzero(configs["split"].eq(name).to_numpy())
         for name in ["train", "validation", "test"]}
print({name: len(idx) for name, idx in split.items()}, "X:", X.shape, "Y:", Y.shape)

## 3. Escala de avaliação e referência constante

A padronização é ajustada **somente no treino**. Alvos sem variação entre políticas no treino recebem a média constante e saem do erro normalizado. Para os demais, o erro conjunto é $\sqrt{\operatorname{média}_{i,k}[(\hat y_{ik}-y_{ik})^2/\sigma_k^2]}$, com $\sigma_k$ calculado nas configurações de treino. Assim, PIB e inflação entram em escalas comparáveis.

In [ ]:
train, valid, test = (split[name] for name in ["train", "validation", "test"])
# Algumas saídas do 1º trimestre são invariantes à política neste experimento.
# Identificá-las só no treino evita dividir pelo desvio ~0 e inflar o ruído da PCA.
training_mean = Y[train].mean(axis=0)
active_targets = np.ptp(Y[train], axis=0) > 1e-10
print("Alvos constantes no treino:", [name for name, active in zip(TARGETS, active_targets) if not active])
Y[:, ~active_targets] = training_mean[~active_targets]
Y_runs[:, ~active_targets] = training_mean[~active_targets]
evaluation_scale = Y[train].std(axis=0, ddof=0)
if np.any(evaluation_scale[active_targets] <= 0):
    raise ValueError("Há alvos ativos sem variação no treino.")
evaluation_scale[~active_targets] = 1.0
def nrmse(y_true, y_pred):
    return float(np.sqrt(np.mean(((y_pred[:, active_targets] - y_true[:, active_targets]) / evaluation_scale[active_targets]) ** 2)))
def metric_table(y_true, y_pred, name):
    rows = []
    for j, metric in enumerate(METRICS):
        error = y_pred[:, j*12:(j+1)*12] - y_true[:, j*12:(j+1)*12]
        active = active_targets[j*12:(j+1)*12]
        rows.append({
            "model": name, "indicator": metric,
            "MAE": float(np.abs(error).mean()),
            "RMSE": float(np.sqrt(np.mean(error**2))),
            "normalized_RMSE": float(np.sqrt(np.mean(
                (error[:, active] / evaluation_scale[j*12:(j+1)*12][active])**2
            ))),
        })
    return pd.DataFrame(rows)
mean_prediction = np.broadcast_to(Y[train].mean(axis=0), Y[valid].shape)
candidate_predictions = {"mean": mean_prediction}
candidate_models = {"mean": {"kind": "mean", "mean": Y[train].mean(axis=0)}}
print("Média constante, NRMSE na validação:", nrmse(Y[valid], mean_prediction))

## 4. Ridge linear e quadrática

A regressão minimiza $\lVert Y_z-\Phi(X_z)B\rVert_F^2+\alpha\lVert B\rVert_F^2$. A forma linear usa as quatro entradas; a quadrática acrescenta quadrados e interações. Escolhemos $\alpha$ com validação cruzada **apenas dentro do treino**, refazendo as escalas em cada dobra.

In [ ]:
def fit_ridge(x, y, degree, alpha):
    x_scaler = StandardScaler().fit(x)
    y_scaler = StandardScaler().fit(y)
    features = PolynomialFeatures(degree=degree, include_bias=False).fit(
        x_scaler.transform(x)
    )
    model = Ridge(alpha=alpha).fit(
        features.transform(x_scaler.transform(x)), y_scaler.transform(y)
    )
    return {"kind": "ridge", "degree": degree, "alpha": alpha,
            "x_scaler": x_scaler, "y_scaler": y_scaler,
            "features": features, "model": model}

def predict_ridge(bundle, x):
    x_features = bundle["features"].transform(bundle["x_scaler"].transform(x))
    return bundle["y_scaler"].inverse_transform(bundle["model"].predict(x_features))

rng = np.random.default_rng(2026)
permuted = rng.permutation(train)
folds = np.array_split(permuted, 5)
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
cv_rows = []
for degree in [1, 2]:
    for alpha in alphas:
        fold_errors = []
        for heldout in folds:
            fitting = np.setdiff1d(train, heldout)
            fitted = fit_ridge(X[fitting], Y[fitting], degree, alpha)
            prediction = predict_ridge(fitted, X[heldout])
            fold_scale = Y[fitting].std(axis=0, ddof=0)
            fold_errors.append(np.sqrt(np.mean(
                ((prediction[:, active_targets] - Y[heldout][:, active_targets])
                 / fold_scale[active_targets])**2
            )))
        cv_rows.append({"degree": degree, "alpha": alpha,
                        "cv_NRMSE": float(np.mean(fold_errors))})
cv_results = pd.DataFrame(cv_rows)
ridge_models = {}
for degree, name in [(1, "ridge_linear"), (2, "ridge_quadratic")]:
    best_alpha = cv_results.query("degree == @degree").sort_values(
        "cv_NRMSE"
    ).iloc[0]["alpha"]
    fitted = fit_ridge(X[train], Y[train], degree, float(best_alpha))
    ridge_models[name] = fitted
    candidate_models[name] = fitted
    candidate_predictions[name] = predict_ridge(fitted, X[valid])
display(cv_results)
display(pd.DataFrame([
    {"model": name, "validation_NRMSE": nrmse(Y[valid], pred)}
    for name, pred in candidate_predictions.items()
]).sort_values("validation_NRMSE"))

## 5. Curva de aprendizado dentro do treino

Em cada repetição, reservamos aproximadamente 10% das configurações de treino para avaliação interna (mínimo de 10). Ajustamos a Ridge quadrática em subconjuntos crescentes; validação e teste externos continuam intocados.

In [ ]:
curve_rows = []
curve_alpha = 1.0  # Fixo antes das divisões internas; evita usar o holdout na escolha.
internal_holdout_size = max(10, round(0.1 * len(train)))
curve_sizes = [n for n in [20, 40, 60, 70, 100, 200, 300, 350] if n <= len(train) - internal_holdout_size]
for repetition in range(5):
    permutation = rng.permutation(train)
    holdout, pool = permutation[:internal_holdout_size], permutation[internal_holdout_size:]
    for size in curve_sizes:
        subset = pool[:size]
        fitted = fit_ridge(X[subset], Y[subset], 2, curve_alpha)
        pred_train = predict_ridge(fitted, X[subset])
        pred_holdout = predict_ridge(fitted, X[holdout])
        curve_rows.append({
            "repetition": repetition, "train_size": size,
            "training_NRMSE": nrmse(Y[subset], pred_train),
            "internal_holdout_NRMSE": nrmse(Y[holdout], pred_holdout),
        })
curve = pd.DataFrame(curve_rows)
display(curve.groupby("train_size")[["training_NRMSE", "internal_holdout_NRMSE"]]
        .agg(["mean", "std"]))
fig, ax = plt.subplots(figsize=(7, 4))
for column, label in [("training_NRMSE", "Treino"), ("internal_holdout_NRMSE", "Avaliação interna")]:
    grouped = curve.groupby("train_size")[column]
    mean, std = grouped.mean(), grouped.std()
    ax.errorbar(mean.index, mean, yerr=std, marker="o", capsize=3, label=label)
ax.set(xlabel="Configurações usadas no ajuste", ylabel="NRMSE",
       title="Curva de aprendizado: Ridge quadrática")
ax.legend()
plt.show()

## 6. PCA + processos gaussianos

Padronizamos $Y$ no treino e estimamos uma PCA. Cada componente $z_q$ recebe um GP Matérn 3/2 com um comprimento de escala por parâmetro fiscal. O ruído observado em $z_q$ é a variância entre sementes dividida pelo número de sementes, com piso de $10^{-4}$. Comparamos 2, 4 e 8 componentes na validação.

In [ ]:
x_scaler_gp = StandardScaler().fit(X[train])
y_scaler_gp = StandardScaler().fit(Y[train])
pca = PCA(n_components=8, svd_solver="full").fit(y_scaler_gp.transform(Y[train]))
scores = pca.transform(y_scaler_gp.transform(Y[train]))
seed_scores = pca.transform(y_scaler_gp.transform(Y_runs))
seed_ids = run_index.get_level_values("configuration_id").to_numpy()
noise_variance = np.stack([
    seed_scores[seed_ids == ids[i]].var(axis=0, ddof=1) / N_SEEDS
    for i in train
])
noise_variance = np.maximum(noise_variance, 1e-4)
score_scale = scores.std(axis=0, ddof=0)
assert np.all(score_scale > 0)
gp_models = []
x_gp = x_scaler_gp.transform(X[train])
for component in range(8):
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
        length_scale=np.ones(4), length_scale_bounds=(0.1, 100.0), nu=1.5
    )
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=noise_variance[:, component] / score_scale[component]**2,
        normalize_y=False, n_restarts_optimizer=0, random_state=2026,
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        gp.fit(x_gp, scores[:, component] / score_scale[component])
    if caught:
        print(f"Componente {component + 1}: {len(caught)} aviso(s); exemplo: {caught[0].message}")
    gp_models.append(gp)

gp_bundle = {
    "kind": "pca_gp", "x_scaler": x_scaler_gp, "y_scaler": y_scaler_gp,
    "pca": pca, "gps": gp_models, "noise_variance": noise_variance,
    "score_scale": score_scale,
}
def predict_gp(bundle, x, components, return_std=False):
    xz = bundle["x_scaler"].transform(x)
    means, variances = [], []
    for component, gp in enumerate(bundle["gps"][:components]):
        mean, std = gp.predict(xz, return_std=True)
        means.append(mean * bundle["score_scale"][component])
        variances.append((std * bundle["score_scale"][component])**2)
    latent_mean = np.column_stack(means)
    scores_full = np.zeros((len(x), bundle["pca"].n_components_))
    scores_full[:, :components] = latent_mean
    standardized_y = bundle["pca"].inverse_transform(scores_full)
    y = bundle["y_scaler"].inverse_transform(standardized_y)
    if not return_std:
        return y
    # Aproximação: GPs independentes + variância de reconstrução da PCA.
    components_sq = bundle["pca"].components_[:components]**2
    latent_var_y = np.column_stack(variances) @ components_sq
    train_scores = bundle["pca"].transform(bundle["y_scaler"].transform(Y[train]))
    train_scores[:, components:] = 0
    train_reconstruction = bundle["pca"].inverse_transform(train_scores)
    truncation_var = np.mean(
        (bundle["y_scaler"].transform(Y[train]) - train_reconstruction)**2,
        axis=0,
    )
    y_std = np.sqrt(latent_var_y + truncation_var) * bundle["y_scaler"].scale_
    return y, y_std

for components in [2, 4, 8]:
    name = f"pca_gp_{components}"
    candidate_models[name] = {**gp_bundle, "components": components}
    candidate_predictions[name] = predict_gp(gp_bundle, X[valid], components)
display(pd.DataFrame([
    {"model": name, "validation_NRMSE": nrmse(Y[valid], pred)}
    for name, pred in candidate_predictions.items()
]).sort_values("validation_NRMSE"))
display(pd.DataFrame({"component": np.arange(1, 9),
                      "explained_variance_ratio": pca.explained_variance_ratio_,
                      "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_)}))

## 7. MLP opcional

Ative esta etapa somente se a Ridge e o GP deixarem padrões claros de erro. A rede usa uma camada pequena, regularização e três inicializações; a média das previsões reduz a variação entre ajustes. O padrão é **desativado** até haver motivo para incluir outro candidato.

In [ ]:
RUN_MLP = False
if RUN_MLP:
    from sklearn.neural_network import MLPRegressor
    mlp_components = 8
    score_scaler = StandardScaler().fit(scores[:, :mlp_components])
    mlps = []
    for random_state in [2026, 2027, 2028]:
        mlp = MLPRegressor(
            hidden_layer_sizes=(16,), activation="tanh", alpha=1.0,
            max_iter=3000, early_stopping=True, n_iter_no_change=100,
            random_state=random_state,
        )
        mlp.fit(x_gp, score_scaler.transform(scores[:, :mlp_components]))
        mlps.append(mlp)
    mlp_bundle = {
        "kind": "pca_mlp", "x_scaler": x_scaler_gp, "y_scaler": y_scaler_gp,
        "pca": pca, "score_scaler": score_scaler,
        "mlps": mlps, "components": mlp_components,
    }
    def predict_mlp(bundle, x):
        xz = bundle["x_scaler"].transform(x)
        z_scaled = np.mean([model.predict(xz) for model in bundle["mlps"]], axis=0)
        z = bundle["score_scaler"].inverse_transform(z_scaled)
        full = np.zeros((len(x), bundle["pca"].n_components_))
        full[:, :bundle["components"]] = z
        return bundle["y_scaler"].inverse_transform(
            bundle["pca"].inverse_transform(full)
        )
    candidate_models["pca_mlp_8"] = mlp_bundle
    candidate_predictions["pca_mlp_8"] = predict_mlp(mlp_bundle, X[valid])
    print("MLP, NRMSE na validação:", nrmse(Y[valid], candidate_predictions["pca_mlp_8"]))

## 8. Escolher modelo pela validação

A regra padrão mantém a melhor Ridge, salvo se outro modelo reduzir o NRMSE conjunto em pelo menos 5% **e** melhorar pelo menos 6 dos 10 indicadores na validação. Esses limites evitam escolher um modelo mais complexo por ganho pequeno em apenas 10 políticas. Se uma variável econômica for prioritária, defina `SELECT_MODEL` explicitamente nesta célula com base apenas na validação.

In [ ]:
validation_scores = pd.DataFrame([
    {"model": name, "validation_NRMSE": nrmse(Y[valid], prediction)}
    for name, prediction in candidate_predictions.items()
]).sort_values("validation_NRMSE").reset_index(drop=True)
validation_indicators = pd.concat([
    metric_table(Y[valid], prediction, name)
    for name, prediction in candidate_predictions.items()
], ignore_index=True)
display(validation_scores)
display(validation_indicators.pivot(index="indicator", columns="model",
                                     values="normalized_RMSE").round(3))
best_ridge = validation_scores[validation_scores["model"].str.startswith("ridge")].iloc[0]["model"]
ridge_score = nrmse(Y[valid], candidate_predictions[best_ridge])
eligible = []
for name, prediction in candidate_predictions.items():
    if name == best_ridge or name == "mean":
        continue
    score = nrmse(Y[valid], prediction)
    indicator_error = validation_indicators.set_index(["model", "indicator"])["normalized_RMSE"]
    improved = sum(indicator_error[name, metric] < indicator_error[best_ridge, metric] for metric in METRICS)
    if score <= 0.95 * ridge_score and improved >= 6:
        eligible.append((score, name))
automatic_choice = min(eligible)[1] if eligible else best_ridge
if nrmse(Y[valid], candidate_predictions["mean"]) < nrmse(Y[valid], candidate_predictions[automatic_choice]):
    automatic_choice = "mean"
SELECT_MODEL = None  # Ex.: "pca_gp_4"; decisão baseada somente na validação.
selected_name = SELECT_MODEL or automatic_choice
if selected_name not in candidate_models:
    raise ValueError(f"Modelo desconhecido: {selected_name}")
selected = candidate_models[selected_name]
print("Modelo selecionado:", selected_name)
print("Validação NRMSE:", nrmse(Y[valid], candidate_predictions[selected_name]))

## 9. Avaliar **uma vez** nas políticas de teste

O teste permanece fora da escolha de modelo. Registramos MAE e RMSE por indicador e trimestre, com o erro padrão entre sementes como referência de ruído.

In [ ]:
def predict_selected(bundle, x):
    if bundle["kind"] == "mean":
        return np.broadcast_to(bundle["mean"], (len(x), len(bundle["mean"]))).copy()
    if bundle["kind"] == "ridge":
        return predict_ridge(bundle, x)
    if bundle["kind"] == "pca_gp":
        return predict_gp(bundle, x, bundle["components"])
    if bundle["kind"] == "pca_mlp":
        return predict_mlp(bundle, x)
    raise ValueError(bundle["kind"])

test_prediction = predict_selected(selected, X[test])
test_nrmse = nrmse(Y[test], test_prediction)
test_indicators = metric_table(Y[test], test_prediction, selected_name)
test_quarter_rows = []
for j, metric in enumerate(METRICS):
    for q, quarter in enumerate(QUARTERS):
        k = j*12 + q
        residual = test_prediction[:, k] - Y[test, k]
        test_quarter_rows.append({
            "indicator": metric, "quarter": quarter,
            "MAE": float(np.abs(residual).mean()),
            "RMSE": float(np.sqrt(np.mean(residual**2))),
            "median_seed_SE": float(np.median(Y_se[test, k])),
        })
test_quarters = pd.DataFrame(test_quarter_rows)
print("Teste NRMSE:", test_nrmse)
display(test_indicators)
display(test_quarters)
test_regions = pd.DataFrame()
test_region_indicators = pd.DataFrame()
pilot_metadata_path = HERE / "experiments/austria-scale-0.001/metadata.toml"
if EXPERIMENT_NAME != "austria-scale-0.001" and pilot_metadata_path.exists():
    with pilot_metadata_path.open("rb") as f:
        pilot_metadata = tomllib.load(f)
    pilot_bounds = pilot_metadata["policy"]
    inner = np.all(np.column_stack([
        (X[test, j] >= pilot_bounds[name]["lower"] - 1e-12) &
        (X[test, j] <= pilot_bounds[name]["upper"] + 1e-12)
        for j, name in enumerate(INPUTS)
    ]), axis=1)
    region_rows, region_tables = [], []
    for label, mask in [("domínio antigo", inner), ("faixas novas", ~inner)]:
        if mask.any():
            region_rows.append({"region": label, "policies": int(mask.sum()),
                                "NRMSE": nrmse(Y[test][mask], test_prediction[mask])})
            region_tables.append(metric_table(Y[test][mask], test_prediction[mask],
                                              selected_name).assign(region=label))
    test_regions = pd.DataFrame(region_rows)
    test_region_indicators = pd.concat(region_tables, ignore_index=True)
    display(test_regions)
if selected["kind"] == "pca_gp":
    _, gp_std = predict_gp(selected, X[test], selected["components"], return_std=True)
    # Incerteza aproximada para a média das sementes; não calibrada.
    typical_mean_noise = np.median(Y_se[train], axis=0)
    sigma = np.sqrt(gp_std**2 + typical_mean_noise[None, :]**2)
    inside = np.abs(Y[test] - test_prediction) <= 1.96 * sigma
    coverage = pd.DataFrame({
        "indicator": METRICS,
        "nominal_95pct_coverage": [
            float(inside[:, j*12:(j+1)*12].mean())
            for j in range(len(METRICS))
        ],
    })
    display(coverage)
    print("Cobertura diagnóstica, não calibrada; confira o número de políticas de teste.")

## 10. Inspecionar trajetórias e limites físicos

O gráfico mostra uma política reservada. As contagens de violações físicas são diagnósticas; o conjunto de indicadores não impõe restrições automaticamente ao ajuste.

In [ ]:
policy_row = 0
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, metric in zip(axes.flat, [
    "real_gdp", "gdp_deflator_inflation", "unemployment_rate", "debt_to_gdp"
]):
    j = METRICS.index(metric)
    column = slice(j*12, (j+1)*12)
    ax.plot(QUARTERS, Y[test[policy_row], column], "o-", label="BeforeIT (média)")
    ax.plot(QUARTERS, test_prediction[policy_row, column], "s--", label="Surrogate")
    ax.set_title(metric)
    ax.set_xlabel("Trimestre")
axes.flat[0].legend()
fig.suptitle(f"Política de teste {ids[test[policy_row]]}")
fig.tight_layout()
plt.show()
physical_checks = {
    "unemployment_outside_0_1": int(np.sum(
        (test_prediction[:, 3*12:4*12] < 0) |
        (test_prediction[:, 3*12:4*12] > 1)
    )),
    "negative_debt_to_gdp": int(np.sum(test_prediction[:, 7*12:8*12] < 0)),
    "negative_positive_level": int(sum(
        np.sum(test_prediction[:, j*12:(j+1)*12] < 0)
        for j in [0, 1, 2, 5, 6, 8, 9]
    )),
}
display(pd.Series(physical_checks, name="predições fora de limites básicos"))

## 11. Salvar modelo, métricas e proveniência

O arquivo `model.joblib` contém o modelo e seus transformadores. A função de previsão abaixo recusa políticas fora dos limites do experimento; o bundle pertence à calibração austríaca na escala `0.001`.

In [ ]:
# experiment_metadata já foi lido e validado na etapa 2.
bounds = {name: experiment_metadata["policy"][name] for name in INPUTS}
def predict_policy(policy):
    """Prever uma política dentro do domínio fiscal, nesta calibração e escala."""
    missing_keys = set(INPUTS) - set(policy)
    if missing_keys:
        raise ValueError(f"Parâmetros ausentes: {sorted(missing_keys)}")
    values = np.array([[float(policy[name]) for name in INPUTS]])
    for name, value in zip(INPUTS, values[0]):
        if not bounds[name]["lower"] <= value <= bounds[name]["upper"]:
            raise ValueError(f"{name}={value} fora de {bounds[name]}")
    return pd.DataFrame(
        predict_selected(selected, values).reshape(len(METRICS), len(QUARTERS)),
        index=METRICS, columns=QUARTERS,
    )

OUTPUT.mkdir(exist_ok=True)
joblib.dump({"selected_name": selected_name, "model": selected,
             "input_names": INPUTS, "output_names": TARGETS,
             "bounds": bounds, "calibration": experiment_metadata["calibration"],
             "active_targets": active_targets, "training_mean": training_mean},
            OUTPUT / "model.joblib")
validation_scores.to_csv(OUTPUT / "validation_scores.csv", index=False)
validation_indicators.to_csv(OUTPUT / "validation_indicators.csv", index=False)
cv_results.to_csv(OUTPUT / "ridge_cv.csv", index=False)
curve.to_csv(OUTPUT / "learning_curve.csv", index=False)
test_indicators.to_csv(OUTPUT / "test_indicators.csv", index=False)
test_quarters.to_csv(OUTPUT / "test_by_quarter.csv", index=False)
if not test_regions.empty:
    test_regions.to_csv(OUTPUT / "test_by_region.csv", index=False)
    test_region_indicators.to_csv(OUTPUT / "test_region_indicators.csv", index=False)
else:
    (OUTPUT / "test_by_region.csv").unlink(missing_ok=True)
    (OUTPUT / "test_region_indicators.csv").unlink(missing_ok=True)
pd.DataFrame(test_prediction, columns=TARGETS).assign(
    configuration_id=ids[test]
).to_csv(OUTPUT / "test_predictions.csv", index=False)
if selected["kind"] == "pca_gp":
    coverage.to_csv(OUTPUT / "gp_test_coverage.csv", index=False)
else:
    (OUTPUT / "gp_test_coverage.csv").unlink(missing_ok=True)
provenance = {
    "experiment_name": EXPERIMENT_NAME,
    "configurations": N_CONFIGS, "seeds_per_configuration": N_SEEDS,
    "selected_model": selected_name,
    "validation_NRMSE": float(nrmse(Y[valid], candidate_predictions[selected_name])),
    "test_NRMSE": float(test_nrmse),
    "splits": {name: ids[index].tolist() for name, index in split.items()},
    "input_names": INPUTS, "output_names": TARGETS,
    "constant_targets": [name for name, active in zip(TARGETS, active_targets) if not active],
    "model_selection": {"minimum_relative_gain": 0.05, "minimum_improved_indicators": 6},
    "calibration": experiment_metadata["calibration"],
    "policy_bounds": bounds,
    "ridge_alphas": {name: float(bundle["alpha"]) for name, bundle in ridge_models.items()},
    "gp_kernels": [str(gp.kernel_) for gp in gp_models],
    "data_sha256": {
        name: hashlib.sha256((DATA / name).read_bytes()).hexdigest()
        for name in files
    },
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
    "limitations": [
        "Válido apenas no domínio fiscal e na calibração/escala gravados.",
        "Política fixa durante os 12 trimestres.",
        f"Teste contém {len(test)} políticas; confirmar com novas simulações.",
    ],
}
(OUTPUT / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print("Artefatos:", OUTPUT)
display(predict_policy(dict(zip(INPUTS, X[0]))).iloc[:3, :4])